In [6]:
from pathlib import Path
import sys
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, load_runs
from src.preprocess import (
    qc_report,
    plot_sanity,
    plot_dt_diagnostics,
    bias_from_first_seconds,
    time_range,
    sensor_channel_checks,
    save_qc_outputs,
    build_dataset_qc_table,
    split_qc_pass_fail,
)

In [7]:
# Import labeling utilities and load label config
from src.labels import load_label_config, label_dataframe

# Load the label definitions from config
labels_config_path = PROJECT_ROOT / "data" / "labels_config.json"
label_configs = load_label_config(labels_config_path)

print(f"Loaded {len(label_configs)} label definitions")
for name, config in label_configs.items():
    print(f"  - {name}: {len(config.ranges)} range(s)")

Loaded 4 label definitions
  - rough_terrain: 1 range(s)
  - smooth_plus_rough: 2 range(s)
  - smooth_terrain: 2 range(s)
  - collision: 1 range(s)


In [8]:
# Load the test_1 run
test_1_dir = PROJECT_ROOT / "data" / "raw" / "test_1"
run_dirs = [d for d in test_1_dir.glob("log_*") if d.is_dir()]
print(f"Found {len(run_dirs)} run(s) in test_1:")
for run_dir in run_dirs:
    print(f"  - {run_dir.name}")

# Load the first (and likely only) run
if run_dirs:
    run = load_run(run_dirs[0], include_pose=True)
    print(f"\nLoaded run: {run.run_id}")
    print(f"  Accelerometer: {len(run.acc)} samples")
    print(f"  Gyroscope: {len(run.gyro)} samples")
    print(f"  Odometry: {len(run.odo)} samples")
    print(f"  Pose: {len(run.pose) if run.pose is not None else 'Not loaded'} samples")
else:
    print("ERROR: No runs found in test_1")
    run = None

Found 1 run(s) in test_1:
  - log_20260216_114652.530

Loaded run: log_20260216_114652.530
  Accelerometer: 145708 samples
  Gyroscope: 147427 samples
  Odometry: 219888 samples
  Pose: 351858 samples


In [9]:
# Apply labels to each sensor
if run is not None:
    run_labeled_sensors = {}
    
    for sensor_name, sensor_df in run.sensors().items():
        labeled_df = label_dataframe(sensor_df, label_configs, time_column='t')
        run_labeled_sensors[sensor_name] = labeled_df
        
        # Count label distribution
        label_counts = labeled_df['label'].value_counts(dropna=False)
        print(f"\n{sensor_name.upper()} label distribution:")
        print(label_counts)


ACC label distribution:
label
NaN                  128559
smooth_terrain         6835
rough_terrain          4564
smooth_plus_rough      3416
collision              2334
Name: count, dtype: int64

GYRO label distribution:
label
NaN                  130278
smooth_terrain         6835
rough_terrain          4564
smooth_plus_rough      3416
collision              2334
Name: count, dtype: int64

ODO label distribution:
label
NaN                  194174
smooth_terrain        10250
rough_terrain          6841
smooth_plus_rough      5123
collision              3500
Name: count, dtype: int64

POSE label distribution:
label
NaN                  310724
smooth_terrain        16400
rough_terrain         10937
smooth_plus_rough      8197
collision              5600
Name: count, dtype: int64


In [12]:
# Sanity check: show samples near label boundaries
if run is not None:
    print("=== Validation: Sample rows near label boundaries ===\n")
    
    # Check a few label transitions in accelerometer data
    acc_labeled = run_labeled_sensors['acc']
    
    # Find samples with different labels
    transitions = []
    for i in range(1, len(acc_labeled)):
        if acc_labeled['label'].iloc[i] != acc_labeled['label'].iloc[i-1]:
            transitions.append(i)
    
    print(f"Found {len(transitions)} label transitions\n")
    
    # Show first few transitions
    for idx, transition_idx in enumerate(transitions[:3]):
        print(f"Transition {idx + 1} (around row {transition_idx}):")
        start = max(0, transition_idx - 2)
        end = min(len(acc_labeled), transition_idx + 3)
        print(acc_labeled[['t', 'ax', 'label']].iloc[start:end])
        print()

=== Validation: Sample rows near label boundaries ===

Found 128564 label transitions

Transition 1 (around row 1):
              t      ax label
0  1.771239e+09  0.0553   NaN
1  1.771239e+09  0.0519   NaN
2  1.771239e+09  0.0587   NaN
3  1.771239e+09  0.0551   NaN

Transition 2 (around row 2):
              t      ax label
0  1.771239e+09  0.0553   NaN
1  1.771239e+09  0.0519   NaN
2  1.771239e+09  0.0587   NaN
3  1.771239e+09  0.0551   NaN
4  1.771239e+09  0.0567   NaN

Transition 3 (around row 3):
              t      ax label
1  1.771239e+09  0.0519   NaN
2  1.771239e+09  0.0587   NaN
3  1.771239e+09  0.0551   NaN
4  1.771239e+09  0.0567   NaN
5  1.771239e+09  0.0570   NaN



In [13]:
# Save labeled run to data/labeled/
from src.io import RunData, save_labeled_run
from dataclasses import replace

if run is not None:
    # Create a new RunData with the labeled sensor DataFrames
    run_labeled = replace(
        run,
        acc=run_labeled_sensors['acc'],
        gyro=run_labeled_sensors['gyro'],
        odo=run_labeled_sensors['odo'],
        pose=run_labeled_sensors.get('pose')
    )
    
    # Save to data/labeled/
    output_root = PROJECT_ROOT / "data" / "labeled"
    save_labeled_run(run_labeled, output_root)
    
    print(f"Labeled data saved to: {output_root / run.run_id}")
    print(f"Files created:")
    for f in (output_root / run.run_id).glob("*.csv"):
        file_lines = len(pd.read_csv(f))
        print(f"  - {f.name} ({file_lines} rows)")

Labeled data saved to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/labeled/log_20260216_114652.530
Files created:


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_56682/1459595740.py:22: DtypeWarning: Columns (0: label) have mixed types. Specify dtype option on import or set low_memory=False.
  file_lines = len(pd.read_csv(f))


  - log_t0_pose.csv (351858 rows)
  - log_t0_gyro_1.csv (147427 rows)
  - log_t0_acc_1.csv (145708 rows)
  - log_t0_encoder_velocity.csv (219888 rows)
